In [86]:
header = ["execID", "problem", "instance", "executable", "full_instance", "status", "exit_code", "real", "time", "user", "system", "memory"]

In [87]:
import re
import os
import sys
import pandas as pd
from matplotlib import pyplot as plt

df = None
for file in os.listdir("."):
    if not file.endswith(".tsv"):
        continue
    df_curr = pd.read_csv(file, sep="\t", names=header, index_col=False)
    df = pd.concat([df, df_curr], ignore_index=True) if not df is None else df_curr

# Computing Metrics

In [88]:
# solved,par1,par10
import numpy as np


df.loc[:, "solved"] = df["status"] == "complete"
# df.loc[:, "par1"] = np.where(
#     df["status"] == "complete",
#     df["real"],
#     1200
# )
# df.loc[:, "par10"] = np.where(
#     df["status"] == "complete",
#     df["real"],
#     1200*10
# )

df.loc[:, "executable"] = df.loc[:, "executable"].str.replace(f"-plain-","-")
df.loc[:, "executable"] = df.loc[:, "executable"].str.replace(r"(amo|eo)wasp-base$",r"\1wasp-base-lg=py", regex=True)

In [89]:
map_execid_to_description = {
    r"firstSubmission": "no_le",
    r"AMO_LE": "le_enc",
    r"EO_LE": "le_enc",
    r"dmpc": "le_enc",
    r"newbench": "no_le",
    r"minfly_sumWph_mpsHph": "minflyopt-no_le"
}

for execId, desc in map_execid_to_description.items():
    mask = (df["execID"].str.contains(execId, regex=True)) & ~(df["executable"].str.contains(r"^(?:clingo|wasp)", regex=True))
    df.loc[mask, "executable"] = df.loc[mask, "executable"] + f":{desc}"

In [90]:
from typing import Dict, List


def _filterDF(df: pd.DataFrame, filterList: List[Dict[str, str]] = {}, positive=True):
    for filter in filterList:
        sliceDf = None
        regexStr = ""
        for column in filter:
            regex = filter[column]
            notStr = "~" if not positive else ""
            regexStr = f" and {notStr}{regex}" if regexStr else regex
            sliceDf = (df[column].str.contains(regex)) if sliceDf is None else sliceDf & (df[column].str.contains(regex))
        print(f"{'Keeping' if positive else 'Removing'} {regexStr} from {column}")
        if positive:
            df = df[sliceDf]
        else:
            df = df[~sliceDf]

    return df


def filterDF(df: pd.DataFrame, positiveFilter: List[Dict[str, str]] = {}, negativeFilter: List[Dict[str, str]] = {}):
    df = _filterDF(df, positiveFilter, True)
    df = _filterDF(df, negativeFilter, False)
    return df


positiveFilter = [
]

negativeFilter = [
{
    "executable": r"amoclingo-lg=c-r=minfly-l=(f|h|t)-smpc=t:no_le",
}
]

df = filterDF(df, positiveFilter, negativeFilter)

Removing amoclingo-lg=c-r=minfly-l=(f|h|t)-smpc=t:no_le from executable


/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/ipykernel_4926/2324020352.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sliceDf = (df[column].str.contains(regex)) if sliceDf is None else sliceDf & (df[column].str.contains(regex))


In [91]:
df["executable"].unique()

array(['eoclingo-lg=c-r=ijcai-l=f-smpc=t:le_enc',
       'eoclingo-lg=c-r=minfly-l=f-smpc=t:le_enc',
       'eoclingo-lg=c-r=minfly-l=h-smpc=t:le_enc',
       'eoclingo-lg=c-r=minfly-l=t-smpc=t:le_enc',
       'eoclingo-lg=c-r=nomin-l=f-smpc=t:le_enc',
       'eoclingo-lg=c-r=nomin-l=h-smpc=t:le_enc',
       'eoclingo-lg=c-r=nomin-l=t-smpc=t:le_enc',
       'amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-no_le',
       'amoclingo-lg=c-r=minfly-l=h-smpc=t:minflyopt-no_le',
       'amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt-no_le',
       'amoclingo-lg=c-r=minfly-l=f-smpc=t:le_enc',
       'amoclingo-lg=c-r=minfly-l=h-smpc=t:le_enc',
       'amoclingo-lg=c-r=minfly-l=t-smpc=t:le_enc',
       'amoclingo-lg=c-r=nomin-l=f-smpc=t:le_enc',
       'amoclingo-lg=c-r=nomin-l=h-smpc=t:le_enc',
       'amoclingo-lg=c-r=nomin-l=t-smpc=t:le_enc',
       'eoclingo-lg=c-r=ijcai-l=f-smpc=t:no_le',
       'eoclingo-lg=c-r=minfly-l=f-smpc=t:no_le',
       'eoclingo-lg=c-r=minfly-l=h-smpc=t:no_le',
      

# Df AMO

In [92]:
df_amo = df.copy()
df_amo.drop(index=df[df["executable"].str.contains(r"^eo|-eo$")].index, inplace=True)
df_amo.loc[:, "executable"] = df_amo["executable"].str.replace("-amo$","", regex=True)
df_amo["problem"].unique()

array(['GraphColouring', 'Knapsack', 'GroupAssignment', 'Nurse',
       'CombinatorialAuctionBestBounds', 'TspDecisional'], dtype=object)

In [93]:
len(df_amo[(df_amo["executable"] == "amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le") & (df_amo["problem"] == "GroupAssignment") & (df_amo["status"] == "outof memory")])

0

In [94]:
solved_kn_amoclingo_nomin = set(df_amo[(df_amo["executable"] == "amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le") & (df_amo["solved"]) & (df_amo["problem"] == "Knapsack")]["instance"].unique())
solved_kn_amoclingo_minfly = set(df_amo[(df_amo["executable"] == "amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt") & (df_amo["solved"]) & (df_amo["problem"] == "Knapsack")]["instance"].unique())
not_solved_kn_amoclingo_minfly_but_solved_amoclingo_nomin = solved_kn_amoclingo_nomin - solved_kn_amoclingo_minfly
# df_amo[(df_amo["executable"] == "amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt") & (df_amo["instance"].isin(not_solved_kn_amoclingo_minfly_but_solved_amoclingo_nomin))]
df_amo[(df_amo["executable"] == "amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le") & (df_amo["instance"].isin(not_solved_kn_amoclingo_minfly_but_solved_amoclingo_nomin))]

,execID,problem,instance,executable,full_instance,status,exit_code,real,time,user,system,memory,solved
30210,firstSubmission,Knapsack,0000-knapsack-10-22288-53089-type1.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0000-knapsack-10-22288-53089-...,complete,10,0.395,0.12,0.11,0.01,30.8,True
30211,firstSubmission,Knapsack,0001-knapsack-10-21042-17651-type1.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0001-knapsack-10-21042-17651-...,complete,10,0.560,0.39,0.38,0.01,32.2,True
30212,firstSubmission,Knapsack,0002-knapsack-10-21756-170352-type1.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0002-knapsack-10-21756-170352...,complete,10,0.395,0.24,0.23,0.01,32.4,True
30213,firstSubmission,Knapsack,0003-knapsack-10-19930-128092-type2.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0003-knapsack-10-19930-128092...,complete,10,0.410,0.24,0.22,0.02,32.3,True
30214,firstSubmission,Knapsack,0004-knapsack-10-21938-2349242-type2.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0004-knapsack-10-21938-234924...,complete,20,0.413,0.13,0.12,0.01,30.8,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
30302,firstSubmission,Knapsack,0092-knapsack-55-116950-829874-type1.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0092-knapsack-55-116950-82987...,complete,10,0.411,0.28,0.27,0.01,32.1,True
30303,firstSubmission,Knapsack,0093-knapsack-55-116052-6934275-type2.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0093-knapsack-55-116052-69342...,complete,20,0.563,0.33,0.32,0.01,32.3,True
30304,firstSubmission,Knapsack,0094-knapsack-55-115450-13897471-type2.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0094-knapsack-55-115450-13897...,complete,20,0.603,0.43,0.40,0.03,32.2,True
30305,firstSubmission,Knapsack,0095-knapsack-55-115946-15569240-type2.asp,amoclingo-lg=c-r=nomin-l=t-smpc=t:no_le,results/Knapsack/0095-knapsack-55-115946-15569...,complete,20,0.464,0.29,0.28,0.01,31.6,True


## Pivot AMO

In [95]:
# pivot_amo = df_amo.pivot_table(index=["executable"],columns=["problem"], values=["solved","par1", "par10"], aggfunc={"solved":"sum", "par1": "mean", "par10": "mean"}, margins=True, margins_name='Total', fill_value=0)
pivot_amo = df_amo.pivot_table(index=["executable"],columns=["problem"], values=["solved"], aggfunc={"solved":"sum"}, margins=True, margins_name='Total', fill_value=None)
pivot_amo = pivot_amo.reorder_levels([1,0], axis=1).sort_index(axis=1)
pivot_amo = pivot_amo.drop(index="Total")
# pivot_amo.loc[:, "solved"].astype(int)

for col in pivot_amo.columns[(pivot_amo.columns.get_level_values(1) == "solved")]:
    pivot_amo[col] = pivot_amo.loc[:, col].astype("Int64")

pivot_amo

problem,CombinatorialAuctionBestBounds,GraphColouring,GroupAssignment,Knapsack,Nurse,Total,TspDecisional
,solved,solved,solved,solved,solved,solved,solved
executable,,,,,,,
amoclingo-base-lg=c:no_le,307,102,307,77,1,1047,253
amoclingo-base-lg=py:no_le,285,95,141,75,0,848,252
amoclingo-lg=c-r=ijcai-l=f-smpc=t:no_le,325,120,307,77,2,1087,256
amoclingo-lg=c-r=minfly-l=f-smpc=f:le_enc,<NA>,<NA>,303,72,0,375,<NA>
amoclingo-lg=c-r=minfly-l=f-smpc=t:le_enc,<NA>,<NA>,302,73,1,376,<NA>
amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-no_le,339,127,305,75,1,1102,255
amoclingo-lg=c-r=minfly-l=h-smpc=f:le_enc,<NA>,<NA>,306,73,1,380,<NA>
amoclingo-lg=c-r=minfly-l=h-smpc=t:le_enc,<NA>,<NA>,305,73,0,378,<NA>


In [96]:
# lazy=false
# reason=nomin
# lang=cpp
# static_mpc=true

import glob


def create_solvers(base_file: str = "base_amo", pivot = pivot_amo, output_dir = "amoclingo_versions"):
    map_key_param = {
    "l": "lazy",
    "r": "reason",
    "lg": "lang",
    "l": "lazy",
    "smpc": "static_mpc"
    }

    map_value = {
    "t": "true",
    "f": "false",
    "h": "hybrid",
    "c": "cpp",
    }

    dir_solvers = "solver_to_move_to_clown"

    base_path = f"{dir_solvers}/{base_file}.sh"
    with open(base_path, "r") as f:
        base_sh = "".join(f.readlines())

    for solver in glob.glob(f"{dir_solvers}/{output_dir}/*"):
        # print(f"removing: {solver}")
        if os.path.isfile(solver):
            os.remove(solver)

    smpc = "t"
    solvers_to_do = set(re.sub(r":.*","",e) for e in list(pivot.index) if not re.search("(-base-|^clingo|wasp|py)", e))
    solvers_to_do = solvers_to_do.union(e.replace("-smpc=t","-smpc=f") for e in solvers_to_do)
    print(f"solvers_to_do: {solvers_to_do}")
    for s in solvers_to_do:
        s = str(s)
        if re.search("(-base-|^clingo|wasp|py)", s): continue
        s = re.sub(r":.*","",s)
        content_s = str(base_sh)
        for key_value in re.findall(r"-\w+=\w+", s):
            key = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("key")
            value = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("value")
            content_s = re.sub(rf"{map_key_param[key]}=\w+", f"{map_key_param[key]}={map_value.get(value, value)}", content_s)
        if "-smpc" in s: s = f"{s}.bash"
        else : s = f"{s}-smpc={smpc}.bash"
        print(f"Writing: {s}")
        with open(f"{dir_solvers}/{output_dir}/{s}", "w") as f:
            print(f"# s: {s}", file=f)
            print(content_s, file=f)

create_solvers(base_file= "base_amo", pivot = pivot_amo, output_dir = "amoclingo_versions")

solvers_to_do: {'amoclingo-lg=c-r=nomin-l=f-smpc=t', 'amoclingo-lg=c-r=ijcai-l=f-smpc=t', 'amoclingo-lg=c-r=nomin-l=t-smpc=t', 'amoclingo-lg=c-r=nomin-l=h-smpc=t', 'amoclingo-lg=c-r=minfly-l=h-smpc=t', 'amoclingo-lg=c-r=minfly-l=f-smpc=t', 'amoclingo-lg=c-r=ijcai-l=f-smpc=f', 'amoclingo-lg=c-r=minfly-l=h-smpc=f', 'amoclingo-lg=c-r=minfly-l=f-smpc=f', 'amoclingo-lg=c-r=minfly-l=t-smpc=f', 'amoclingo-lg=c-r=nomin-l=t-smpc=f', 'amoclingo-lg=c-r=nomin-l=h-smpc=f', 'amoclingo-lg=c-r=nomin-l=f-smpc=f', 'amoclingo-lg=c-r=minfly-l=t-smpc=t'}
Writing: amoclingo-lg=c-r=nomin-l=f-smpc=t.bash
Writing: amoclingo-lg=c-r=ijcai-l=f-smpc=t.bash
Writing: amoclingo-lg=c-r=nomin-l=t-smpc=t.bash
Writing: amoclingo-lg=c-r=nomin-l=h-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=h-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=f-smpc=t.bash
Writing: amoclingo-lg=c-r=ijcai-l=f-smpc=f.bash
Writing: amoclingo-lg=c-r=minfly-l=h-smpc=f.bash
Writing: amoclingo-lg=c-r=minfly-l=f-smpc=f.bash
Writing: amoclingo-lg=c-

In [97]:
def create_cactus_df(df_input: pd.DataFrame) -> pd.DataFrame:
    return pd.concat(
        {
            solver: df_input[(df_input["executable"] == solver) & (df_input["solved"])]["real"]
                    .sort_values()
                    .reset_index(drop=True)
            for solver in df_input["executable"].unique()
        },
        axis=1
    )

In [98]:
df_amo

,execID,problem,instance,executable,full_instance,status,exit_code,real,time,user,system,memory,solved
3185,AIJ@2026-06-21-12-56:minfly_sumWph_mpsHph,GraphColouring,0001-graph_colouring-125-0_1200.asp,amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-n...,results/GraphColouring/0001-graph_colouring-12...,complete,10,0.928,0.68,0.65,0.03,52.9,True
3186,AIJ@2026-06-21-12-56:minfly_sumWph_mpsHph,GraphColouring,0001-graph_colouring-125-0_2400.asp,amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-n...,results/GraphColouring/0001-graph_colouring-12...,outof time,143,1200.275,1200.09,1199.78,0.31,308.4,False
3187,AIJ@2026-06-21-12-56:minfly_sumWph_mpsHph,GraphColouring,0001-graph_colouring-125-0_3600.asp,amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-n...,results/GraphColouring/0001-graph_colouring-12...,outof time,143,1200.657,1200.49,1200.24,0.25,272.3,False
3188,AIJ@2026-06-21-12-56:minfly_sumWph_mpsHph,GraphColouring,0001-graph_colouring-125-0_4800.asp,amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-n...,results/GraphColouring/0001-graph_colouring-12...,complete,20,1165.708,1164.51,1164.36,0.15,173.4,True
3189,AIJ@2026-06-21-12-56:minfly_sumWph_mpsHph,GraphColouring,0001-graph_colouring-125-0_6000.asp,amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-n...,results/GraphColouring/0001-graph_colouring-12...,complete,20,2.541,2.19,2.14,0.05,57.9,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
52045,AIJ@2026-06-03-18-21:firstSubmissionBench,Nurse,10nurses.asp,amowasp-base-lg=py:no_le,results/Nurse/10nurses.asp.out,outof time,143,1200.522,1200.26,1200.13,0.13,185.6,False
52046,AIJ@2026-06-03-18-21:firstSubmissionBench,Nurse,164nurses.asp,amowasp-base-lg=py:no_le,results/Nurse/164nurses.asp.out,outof memory,143,105.165,104.15,98.93,5.22,8200.3,False
52047,AIJ@2026-06-03-18-21:firstSubmissionBench,Nurse,20nurses.asp,amowasp-base-lg=py:no_le,results/Nurse/20nurses.asp.out,outof time,143,1201.155,1201.00,1200.73,0.27,389.9,False
52048,AIJ@2026-06-03-18-21:firstSubmissionBench,Nurse,41nurses.asp,amowasp-base-lg=py:no_le,results/Nurse/41nurses.asp.out,outof time,143,1200.862,1200.41,1197.90,2.51,1149.8,False


In [99]:
df_cactus_amo = create_cactus_df(df_amo)
df_cactus_amo.to_csv("plots/catcus_amo.csv")
df_cactus_amo

,amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-no_le,amoclingo-lg=c-r=minfly-l=h-smpc=t:minflyopt-no_le,amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt-no_le,amoclingo-lg=c-r=minfly-l=f-smpc=t:le_enc,amoclingo-lg=c-r=minfly-l=h-smpc=t:le_enc,amoclingo-lg=c-r=minfly-l=t-smpc=t:le_enc,amoclingo-lg=c-r=nomin-l=f-smpc=t:le_enc,amoclingo-lg=c-r=nomin-l=h-smpc=t:le_enc,amoclingo-lg=c-r=nomin-l=t-smpc=t:le_enc,amoclingo-base-lg=c:no_le,...,amoclingo-lg=c-r=nomin-l=h-smpc=f:le_enc,amoclingo-lg=c-r=nomin-l=t-smpc=f:le_enc,amoclingo-lg=py-r=nomin-l=f-smpc=t:no_le,amoclingo-lg=py-r=nomin-l=t-smpc=t:no_le,amoclingo-lg=py-r=nomin-l=h-smpc=t:no_le,amowasp-lg=py-r=nomin-l=f-smpc=t:no_le,amowasp-lg=py-r=nomin-l=t-smpc=t:no_le,amowasp-lg=py-r=nomin-l=h-smpc=t:no_le,amoclingo-lg=py-r=ijcai-l=f-smpc=t:no_le,amowasp-lg=py-r=ijcai-l=f-smpc=t:no_le
0,0.421,0.452,0.427,0.433,0.538,0.424,0.448,0.415,0.423,0.310,...,0.442,0.436,0.304,0.380,0.392,0.403,0.388,0.404,0.320,0.392
1,0.483,0.459,0.439,0.553,0.552,0.439,0.537,0.429,0.430,0.393,...,0.442,0.444,0.387,0.396,0.395,0.405,0.391,0.405,0.397,0.397
2,0.541,0.550,0.442,0.557,0.559,0.557,0.558,0.431,0.441,0.395,...,0.551,0.535,0.397,0.402,0.401,0.405,0.394,0.405,0.399,0.398
3,0.549,0.553,0.444,0.561,0.561,0.563,0.559,0.557,0.544,0.395,...,0.553,0.547,0.402,0.403,0.402,0.407,0.398,0.418,0.401,0.399
4,0.551,0.558,0.542,0.562,0.561,0.572,0.564,0.557,0.551,0.400,...,0.556,0.552,0.403,0.404,0.403,0.407,0.400,0.420,0.402,0.401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1182,NaN,NaN,1038.408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1183,NaN,NaN,1083.184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1184,NaN,NaN,1153.687,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1185,NaN,NaN,1157.782,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
df_cactus_amo["amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt"]

KeyError: 'amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt'

# Df EO

In [ ]:
df_eo = df.copy()
df_eo.drop(index=df[df["executable"].str.contains("(^amo|-amo$)")].index, inplace=True)
df_eo.loc[:, "executable"] = df_eo["executable"].str.replace(r"-eo$","", regex=True)
df_eo

/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/ipykernel_4926/3482083397.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_eo.drop(index=df[df["executable"].str.contains("(^amo|-amo$)")].index, inplace=True)


,execID,problem,instance,executable,full_instance,status,exit_code,real,time,user,system,memory,solved
0,AIJ@2026-06-08-10-09:EO_LE,Knapsack,0000-knapsack-10-22288-53089-type1.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:le_enc,results/Knapsack/0000-knapsack-10-22288-53089-...,complete,10,0.685,0.48,0.45,0.03,36.7,True
1,AIJ@2026-06-08-10-09:EO_LE,Knapsack,0001-knapsack-10-21042-17651-type1.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:le_enc,results/Knapsack/0001-knapsack-10-21042-17651-...,complete,10,0.563,0.37,0.33,0.04,36.8,True
2,AIJ@2026-06-08-10-09:EO_LE,Knapsack,0002-knapsack-10-21756-170352-type1.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:le_enc,results/Knapsack/0002-knapsack-10-21756-170352...,complete,10,0.567,0.38,0.35,0.03,36.7,True
3,AIJ@2026-06-08-10-09:EO_LE,Knapsack,0003-knapsack-10-19930-128092-type2.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:le_enc,results/Knapsack/0003-knapsack-10-19930-128092...,complete,10,0.689,0.40,0.37,0.03,36.4,True
4,AIJ@2026-06-08-10-09:EO_LE,Knapsack,0004-knapsack-10-21938-2349242-type2.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:le_enc,results/Knapsack/0004-knapsack-10-21938-234924...,complete,20,0.544,0.37,0.32,0.05,36.4,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
53395,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,345-group-assignment-250-35-middle.asp,eowasp-base-lg=py:no_le,results/GroupAssignment/345-group-assignment-2...,outof time,143,1200.881,1200.22,1195.01,5.21,2740.1,False
53396,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,346-group-assignment-250-35-middle.asp,eowasp-base-lg=py:no_le,results/GroupAssignment/346-group-assignment-2...,outof time,143,1201.541,1201.05,1197.17,3.88,2070.8,False
53397,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,347-group-assignment-250-35-middle.asp,eowasp-base-lg=py:no_le,results/GroupAssignment/347-group-assignment-2...,outof time,143,1201.562,1200.56,1195.78,4.78,2317.0,False
53398,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,348-group-assignment-250-35-punsat.asp,eowasp-base-lg=py:no_le,results/GroupAssignment/348-group-assignment-2...,outof time,143,1201.375,1200.98,1196.50,4.48,2626.3,False


In [ ]:
df_cactus_eo = create_cactus_df(df_eo)
df_cactus_eo.to_csv("plots/catcus_eo.csv")

## Pivot EO

In [ ]:
# pivot_eo = df_eo.pivot_table(index=["executable"],columns=["problem"], values=["solved","par1", "par10"], aggfunc={"solved":"sum", "par1": "mean", "par10": "mean"}, margins=True, margins_name='Total', fill_value=0)
pivot_eo = df_eo.pivot_table(index=["executable"],columns=["problem"], values=["solved"], aggfunc={"solved":"sum"}, margins=True, margins_name='Total', fill_value=None)
pivot_eo = pivot_eo.reorder_levels([1,0], axis=1).sort_index(axis=1)
pivot_eo = pivot_eo.drop(index="Total")

for col in pivot_eo.columns[(pivot_eo.columns.get_level_values(1) == "solved")]:
    pivot_eo[col] = pivot_eo.loc[:, col].astype("Int64")

pivot_eo

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total,TspDecisional
,solved,solved,solved,solved,solved,solved
executable,,,,,,
clingo,95,350,47,4,625,129
eoclingo-base-lg=c:no_le,101,350,77,1,785,256
eoclingo-base-lg=py:no_le,79,280,74,0,686,253
eoclingo-lg=c-r=ijcai-l=f-smpc=t:le_enc,<NA>,350,82,1,433,<NA>
eoclingo-lg=c-r=ijcai-l=f-smpc=t:no_le,153,350,77,1,839,258
eoclingo-lg=c-r=minfly-l=f-smpc=f:le_enc,<NA>,350,82,0,432,<NA>
eoclingo-lg=c-r=minfly-l=f-smpc=t:le_enc,<NA>,350,82,0,432,<NA>
eoclingo-lg=c-r=minfly-l=f-smpc=t:no_le,131,350,74,0,810,255


In [ ]:
create_solvers(base_file= "base_eo", pivot = pivot_eo, output_dir = "eoclingo_versions")

solvers_to_do: {'eoclingo-lg=c-r=nomin-l=f-smpc=f', 'eoclingo-lg=c-r=minfly-l=t-smpc=f', 'eoclingo-lg=c-r=nomin-l=h-smpc=f', 'eoclingo-lg=c-r=minfly-l=h-smpc=t', 'eoclingo-lg=c-r=minfly-l=t-smpc=t', 'eoclingo-lg=c-r=nomin-l=h-smpc=t', 'eoclingo-lg=c-r=nomin-l=f-smpc=t', 'eoclingo-lg=c-r=minfly-l=h-smpc=f', 'eoclingo-lg=c-r=ijcai-l=f-smpc=t', 'eoclingo-lg=c-r=minfly-l=f-smpc=t', 'eoclingo-lg=c-r=nomin-l=t-smpc=t', 'eoclingo-lg=c-r=minfly-l=f-smpc=f', 'eoclingo-lg=c-r=ijcai-l=f-smpc=f', 'eoclingo-lg=c-r=nomin-l=t-smpc=f'}
Writing: eoclingo-lg=c-r=nomin-l=f-smpc=f.bash
Writing: eoclingo-lg=c-r=minfly-l=t-smpc=f.bash
Writing: eoclingo-lg=c-r=nomin-l=h-smpc=f.bash
Writing: eoclingo-lg=c-r=minfly-l=h-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=h-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=f-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=h-smpc=f.bash
Writing: eoclingo-lg=c-r=ijcai-l=f-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=f-smpc=t.bash

# Latex Functions

In [ ]:
import subprocess
from pathlib import Path
import shutil

def create_latex_visualization(latex_input: str, output_dir: str, output_name, tex_name: str = "main"):
    latex = rf"""
            \documentclass{{article}}
            \usepackage{{booktabs}}
            \usepackage{{multirow}}
            \pagestyle{{empty}}
            \begin{{document}}
            {latex_input}
            \end{{document}}
            """
    output_path = Path(f"{output_dir}/{output_name}")
    output_path.mkdir(parents=True, exist_ok=True)
    tex_name = f"{tex_name}.tex"
    tex_file = output_path / tex_name
    tex_file.write_text(latex)
    subprocess.run(
        ["/Library/TeX/texbin/pdflatex", tex_name],
        cwd=output_path
    )

## Cactus

In [116]:
def create_cactus_latex(file: str, legend_style: str = "at={(1.55,1.0)},anchor=north,fill=none", xmin=0, xmax=None, ymin=0, ymax=1200, subset_solvers = None):
    
    df_cac = pd.read_csv(file, sep=",", header="infer")
    def get_style_plot(solver: str):
        style = {}

        if re.search("^(eo|amo)clingo", solver):
            style["color"] = "amoclingo"
        elif re.search("^(eo|amo)wasp", solver):
            style["color"] = "amowasp"
        elif re.search("^clingo", solver):
            style["color"] = "clingo"
        elif re.search("^wasp", solver):
            style["color"] = "wasp"
        else:
            raise Exception(f"Invalid Solver: {solver}")
        
        if re.search("(eo|amo)clingo",solver):
            style["mark"] = "o"
        elif re.search("(eo|amo)wasp", solver):
            style["mark"] = "o"
        elif re.search("^clingo", solver):
            style["mark"] = "+"
        elif re.search("^wasp", solver):
            style["mark"] = "+"
        else:
            raise Exception(f"Invalid Solver: {solver}")
        
        return style 
    
    plot_lines = []
    columns = list(df_cac.columns)
    for i in range(1, len(columns)):
        solver = columns[i]
        if not subset_solvers is None and not solver in subset_solvers: continue
        plot_line = []
        style = get_style_plot(solver)
        plot_line = f"""
            \\addplot [mark size=2pt, color={style['color']}, mark={style['mark']}] [unbounded coords=jump] table[col sep=comma, y index={i}] {{./{file}}};
            \\addlegendentry{{{re.sub(r'(_)',r'\\\1', solver)}}}
            """
        plot_lines.append(plot_line)

    xmax_str = f"xmax={xmax}" if xmax else ""
    tex_catctus = f"""  
    \\begin{{tikzpicture}}[scale=0.7]
        \\pgfkeys{{/pgf/number format/set thousands separator = {{}}}}
        \\begin{{axis}}[
        scale only axis
        , xlabel={{Solved instances}}
        , ylabel={{Time (s)}}    
        , xmin=0, {xmax_str}
        , ymin=0, ymax=1220
        , legend style={{{legend_style}}}
        , legend columns=1
        , width=0.65\\textwidth
        , height=0.40\\textwidth
        , major tick length=2pt
        , title= {{Catctus Plot}}
        ]

        {'\n'.join(plot_lines)}            

        \\end{{axis}}
    \\end{{tikzpicture}}%
    """
    return tex_catctus

catcus_latex_amo = create_cactus_latex("plots/catcus_amo.csv", subset_solvers=["clingo", "wasp", "amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt-no_le", "amowasp-base-lg=py:no_le"])
print(catcus_latex_amo)

  
    \begin{tikzpicture}[scale=0.7]
        \pgfkeys{/pgf/number format/set thousands separator = {}}
        \begin{axis}[
        scale only axis
        , xlabel={Solved instances}
        , ylabel={Time (s)}    
        , xmin=0, 
        , ymin=0, ymax=1220
        , legend style={at={(1.55,1.0)},anchor=north,fill=none}
        , legend columns=1
        , width=0.65\textwidth
        , height=0.40\textwidth
        , major tick length=2pt
        , title= {Catctus Plot}
        ]

        
            \addplot [mark size=2pt, color=amoclingo, mark=o] [unbounded coords=jump] table[col sep=comma, y index=3] {./plots/catcus_amo.csv};
            \addlegendentry{amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt-no\_le}
            

            \addplot [mark size=2pt, color=amowasp, mark=o] [unbounded coords=jump] table[col sep=comma, y index=16] {./plots/catcus_amo.csv};
            \addlegendentry{amowasp-base-lg=py:no\_le}
            

            \addplot [mark size=2pt, color=cli

In [117]:
catcus_latex_eo = create_cactus_latex("plots/catcus_eo.csv", subset_solvers=["clingo", "wasp", "eoclingo-lg=c-r=minfly-l=t-smpc=t:no_le", "eowasp-base-lg=py:no_le"])
print(catcus_latex_eo)

  
    \begin{tikzpicture}[scale=0.7]
        \pgfkeys{/pgf/number format/set thousands separator = {}}
        \begin{axis}[
        scale only axis
        , xlabel={Solved instances}
        , ylabel={Time (s)}    
        , xmin=0, 
        , ymin=0, ymax=1220
        , legend style={at={(1.55,1.0)},anchor=north,fill=none}
        , legend columns=1
        , width=0.65\textwidth
        , height=0.40\textwidth
        , major tick length=2pt
        , title= {Catctus Plot}
        ]

        
            \addplot [mark size=2pt, color=amoclingo, mark=o] [unbounded coords=jump] table[col sep=comma, y index=11] {./plots/catcus_eo.csv};
            \addlegendentry{eoclingo-lg=c-r=minfly-l=t-smpc=t:no\_le}
            

            \addplot [mark size=2pt, color=clingo, mark=+] [unbounded coords=jump] table[col sep=comma, y index=15] {./plots/catcus_eo.csv};
            \addlegendentry{clingo}
            

            \addplot [mark size=2pt, color=amowasp, mark=o] [unbounded coords=

## Tables

In [ ]:
def get_pivot_latex(pivot: pd.DataFrame):
    pivot_latex = pivot.copy()
    pivot_latex.index = pivot_latex.index.str.replace("_", "\\_", regex=False)
    for c in pivot_latex.columns:
        maxVal = pivot_latex[c].max()
        pivot_latex[c] = pivot_latex[c].astype(object)
        for r in pivot_latex.index:
            v = pivot_latex.at[r, c]
            if pd.notna(maxVal) and pd.notna(v) and v == maxVal:
                pivot_latex.at[r, c] = rf"\textbf{{{v}}}"
            else:
                pivot_latex.at[r, c] = str(v)
    display(pivot_latex)
    latex = pivot_latex.to_latex(
        multicolumn=True,           
        multicolumn_format='c',     
        multirow=True,              
        bold_rows=False,    
        na_rep='-',                 
        label="tab:results",
        position="t!",
        escape=False,               
    )
    return latex

table_amo_res = get_pivot_latex(pivot_amo)
print(table_amo_res)
create_latex_visualization(table_amo_res, "tables", "table_amo")

problem,CombinatorialAuctionBestBounds,GraphColouring,GroupAssignment,Knapsack,Nurse,Total,TspDecisional
,solved,solved,solved,solved,solved,solved,solved
executable,,,,,,,
amoclingo-base-lg=c:no\_le,307,102,307,\textbf{77},1,1047,253
amoclingo-base-lg=py:no\_le,285,95,141,75,0,848,252
amoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le,325,120,307,\textbf{77},2,1087,256
amoclingo-lg=c-r=minfly-l=f-smpc=f:le\_enc,<NA>,<NA>,303,72,0,375,<NA>
amoclingo-lg=c-r=minfly-l=f-smpc=t:le\_enc,<NA>,<NA>,302,73,1,376,<NA>
amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt,\textbf{339},127,305,75,1,1102,255
amoclingo-lg=c-r=minfly-l=h-smpc=f:le\_enc,<NA>,<NA>,306,73,1,380,<NA>
amoclingo-lg=c-r=minfly-l=h-smpc=t:le\_enc,<NA>,<NA>,305,73,0,378,<NA>


\begin{table}[t!]
\label{tab:results}
\begin{tabular}{llllllll}
\toprule
problem & CombinatorialAuctionBestBounds & GraphColouring & GroupAssignment & Knapsack & Nurse & Total & TspDecisional \\
 & solved & solved & solved & solved & solved & solved & solved \\
executable &  &  &  &  &  &  &  \\
\midrule
amoclingo-base-lg=c:no\_le & 307 & 102 & 307 & \textbf{77} & 1 & 1047 & 253 \\
amoclingo-base-lg=py:no\_le & 285 & 95 & 141 & 75 & 0 & 848 & 252 \\
amoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le & 325 & 120 & 307 & \textbf{77} & 2 & 1087 & 256 \\
amoclingo-lg=c-r=minfly-l=f-smpc=f:le\_enc & <NA> & <NA> & 303 & 72 & 0 & 375 & <NA> \\
amoclingo-lg=c-r=minfly-l=f-smpc=t:le\_enc & <NA> & <NA> & 302 & 73 & 1 & 376 & <NA> \\
amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt & \textbf{339} & 127 & 305 & 75 & 1 & 1102 & 255 \\
amoclingo-lg=c-r=minfly-l=h-smpc=f:le\_enc & <NA> & <NA> & 306 & 73 & 1 & 380 & <NA> \\
amoclingo-lg=c-r=minfly-l=h-smpc=t:le\_enc & <NA> & <NA> & 305 & 73 & 0 & 378 & <NA> \\
amoc

In [ ]:
table_eo_res = get_pivot_latex(pivot_eo)
print(table_eo_res)
create_latex_visualization(table_eo_res, "tables", "table_eo")

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total,TspDecisional
,solved,solved,solved,solved,solved,solved
executable,,,,,,
clingo,95,\textbf{350},47,\textbf{4},625,129
eoclingo-base-lg=c:no\_le,101,\textbf{350},77,1,785,256
eoclingo-base-lg=py:no\_le,79,280,74,0,686,253
eoclingo-lg=c-r=ijcai-l=f-smpc=t:le\_enc,<NA>,\textbf{350},82,1,433,<NA>
eoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le,153,\textbf{350},77,1,839,258
eoclingo-lg=c-r=minfly-l=f-smpc=f:le\_enc,<NA>,\textbf{350},82,0,432,<NA>
eoclingo-lg=c-r=minfly-l=f-smpc=t:le\_enc,<NA>,\textbf{350},82,0,432,<NA>
eoclingo-lg=c-r=minfly-l=f-smpc=t:no\_le,131,\textbf{350},74,0,810,255


\begin{table}[t!]
\label{tab:results}
\begin{tabular}{lllllll}
\toprule
problem & GraphColouring & GroupAssignment & Knapsack & Nurse & Total & TspDecisional \\
 & solved & solved & solved & solved & solved & solved \\
executable &  &  &  &  &  &  \\
\midrule
clingo & 95 & \textbf{350} & 47 & \textbf{4} & 625 & 129 \\
eoclingo-base-lg=c:no\_le & 101 & \textbf{350} & 77 & 1 & 785 & 256 \\
eoclingo-base-lg=py:no\_le & 79 & 280 & 74 & 0 & 686 & 253 \\
eoclingo-lg=c-r=ijcai-l=f-smpc=t:le\_enc & <NA> & \textbf{350} & 82 & 1 & 433 & <NA> \\
eoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le & 153 & \textbf{350} & 77 & 1 & 839 & 258 \\
eoclingo-lg=c-r=minfly-l=f-smpc=f:le\_enc & <NA> & \textbf{350} & 82 & 0 & 432 & <NA> \\
eoclingo-lg=c-r=minfly-l=f-smpc=t:le\_enc & <NA> & \textbf{350} & 82 & 0 & 432 & <NA> \\
eoclingo-lg=c-r=minfly-l=f-smpc=t:no\_le & 131 & \textbf{350} & 74 & 0 & 810 & 255 \\
eoclingo-lg=c-r=minfly-l=h-smpc=f:le\_enc & <NA> & \textbf{350} & 81 & 0 & 431 & <NA> \\
eoclingo-lg=c-r=minfl

# Paper Tables

In [ ]:
# template_table =  r"""
#     \begin{tabular}{llrrrrr}
#     \toprule
#     &  & \textbf{WGC} & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{Total}\\
#     \midrule
#     & \#Instances & 300 & 350 & 100 & 5 & 755\\
#     \midrule
#     \multirow{3}{*}{\rotatebox{90}{\textsc{amo}}}
#     & \amowaspij & 64 & 65 & 71 & 0 & 200\\
#     & \amoclingoijpy & 95 & 141 & 75 & 0 & 311\\
#     & \amoclingoijc & \textbf{102} & \textbf{307} & \textbf{77} & \textbf{2} & \textbf{488}\\
#     \midrule
#     \multirow{3}{*}{\rotatebox{90}{\textsc{eo}}} 
#     & \eowaspij & 50 & 130 & 73 & 0 & 253\\
#     & \eoclingoijpy & 79 & 153 & 74 & 0 & 306\\
#     & \eoclingoijc & \textbf{100} & \textbf{311} & \textbf{76} & \textbf{1} & \textbf{488}\\
#     \bottomrule
#     \end{tabular}
# """

def create_paper_table(executables = r"-base-", benchmarks = ["WGC", "GA", "KC", "NS", "TSP", "CA"]):
    
    template_table =  r"""
    \begin{tabular}{llplaceholder_direction_columns_benchmarksr}
    \toprule
    &  & place_holder_benchmarks & \textbf{Total}\\
    \midrule
    & \#Instances & place_holder_num_instances_per_bench\\
    \midrule
    \multirow{place_holder_amo_num_bench}{*}{\rotatebox{90}{\textsc{amo}}}
    place_holder_amo
    \midrule
    \multirow{place_holder_eo_num_bench}{*}{\rotatebox{90}{\textsc{eo}}} 
    place_holder_eo
    \bottomrule
    \end{tabular}
    """

    map_bench_to_benchdf_name = {
        "WGC": "GraphColouring",
        "GA": "GroupAssignment",
        "KC": "Knapsack",
        "NS": "Nurse",
        "TSP": "TspDecisional",
        "CA": "CombinatorialAuctionBestBounds"
    }
    amo_mask = df["executable"].str.contains(r"(?:^amo|-amo$)", regex=True)
    eo_mask = ~df["executable"].str.contains(r"(?:^amo|-amo$)", regex=True)
    selected_executables_mask = df["executable"].str.contains(executables, regex=True)

    selected_benchmarks_mask = df["problem"].isin([map_bench_to_benchdf_name[b] for b in benchmarks])
    
    template_table = template_table.replace("placeholder_direction_columns_benchmarks", "r" * len(benchmarks))
    template_table = template_table.replace("place_holder_benchmarks", " & ".join(fr"\textbf{{{b}}}" for b in benchmarks))

    num_instances = [len(df[df["problem"] == map_bench_to_benchdf_name[b]]["instance"].unique()) for b in benchmarks] + [len(df["instance"].unique())] 
    template_table = template_table.replace("place_holder_num_instances_per_bench", " & ".join(str(n) for n in num_instances))

    n_exe_amo = len(df[amo_mask & selected_executables_mask]["executable"].unique())
    n_exe_eo = len(df[eo_mask & selected_executables_mask]["executable"].unique())

    template_table = template_table.replace("place_holder_amo_num_bench", f"{n_exe_amo}")
    template_table = template_table.replace("place_holder_eo_num_bench", f"{n_exe_eo}")

    def create_rows(subset_mask):
        subset_mask &= selected_benchmarks_mask
        bench_max_solved: dict[str, int] = {}
        all_bench_regex = r".*"
        for b in benchmarks + [all_bench_regex]:
            bench_mask = df["problem"].str.contains(map_bench_to_benchdf_name.get(b, b), regex=True)
            values = []
            for e in df[subset_mask & selected_executables_mask]["executable"].unique():
                executable_mask = df["executable"] == e
                solved = df[subset_mask & executable_mask & bench_mask]["solved"].sum()
                values.append(solved)
            max_value = max(values)
            bench_max_solved[b] = max_value

        rows = []
        for e in df[subset_mask & selected_executables_mask]["executable"].unique():
            row = ["", re.sub(r"(_)",r"\\\1", e)]
            executable_mask = df["executable"] == e
            for b in benchmarks + [all_bench_regex]:
                bench_mask = df["problem"].str.contains(map_bench_to_benchdf_name.get(b, b), regex=True)
                nsolved = len(df[subset_mask & executable_mask & bench_mask]["solved"])
                if nsolved == 0:
                    row.append(r"-")
                    continue
                solved = df[subset_mask & executable_mask & bench_mask]["solved"].sum()
                row.append(fr"\textbf{{{solved}}}" if solved == bench_max_solved[b] else str(solved))
            rows.append(row)
        return rows

    rows_amo = create_rows(amo_mask)
    rows_string = "\n".join([" & ".join(row) + r"\\"  for row in rows_amo])
    template_table = template_table.replace("place_holder_amo", rows_string)

    rows_eo = create_rows(eo_mask)
    rows_string = "\n".join([" & ".join(row) + r"\\"  for row in rows_eo])
    template_table = template_table.replace("place_holder_eo", rows_string)
    
    print(template_table)



In [ ]:
create_paper_table(executables = r"-base-")


    \begin{tabular}{llrrrrrrr}
    \toprule
    &  & \textbf{WGC} & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{TSP} & \textbf{CA} & \textbf{Total}\\
    \midrule
    & \#Instances & 300 & 350 & 100 & 5 & 300 & 340 & 1395\\
    \midrule
    \multirow{3}{*}{\rotatebox{90}{\textsc{amo}}}
     & amoclingo-base-lg=c:no\_le & \textbf{102} & \textbf{307} & \textbf{77} & \textbf{1} & \textbf{253} & \textbf{307} & \textbf{1047}\\
 & amoclingo-base-lg=py:no\_le & 95 & 141 & 75 & 0 & 252 & 285 & 848\\
 & amowasp-base-lg=py:no\_le & 64 & 65 & 71 & 0 & 131 & 306 & 637\\
    \midrule
    \multirow{3}{*}{\rotatebox{90}{\textsc{eo}}} 
     & eoclingo-base-lg=c:no\_le & \textbf{101} & \textbf{350} & \textbf{77} & \textbf{1} & \textbf{256} & - & \textbf{785}\\
 & eoclingo-base-lg=py:no\_le & 79 & 280 & 74 & 0 & 253 & - & 686\\
 & eowasp-base-lg=py:no\_le & 50 & 258 & 73 & 0 & 250 & - & 631\\
    \bottomrule
    \end{tabular}
    


In [ ]:
create_paper_table(executables=r"(?:clingo-base-lg=c|clingo-lg=c-r=ijcai-l=f-smpc=t:no_le)")


    \begin{tabular}{llrrrrrrr}
    \toprule
    &  & \textbf{WGC} & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{TSP} & \textbf{CA} & \textbf{Total}\\
    \midrule
    & \#Instances & 300 & 350 & 100 & 5 & 300 & 340 & 1395\\
    \midrule
    \multirow{2}{*}{\rotatebox{90}{\textsc{amo}}}
     & amoclingo-base-lg=c:no\_le & 102 & \textbf{307} & \textbf{77} & 1 & 253 & 307 & 1047\\
 & amoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le & \textbf{120} & \textbf{307} & \textbf{77} & \textbf{2} & \textbf{256} & \textbf{325} & \textbf{1087}\\
    \midrule
    \multirow{2}{*}{\rotatebox{90}{\textsc{eo}}} 
     & eoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le & \textbf{153} & \textbf{350} & \textbf{77} & \textbf{1} & \textbf{258} & - & \textbf{839}\\
 & eoclingo-base-lg=c:no\_le & 101 & \textbf{350} & \textbf{77} & \textbf{1} & 256 & - & 785\\
    \bottomrule
    \end{tabular}
    


In [ ]:
create_paper_table(executables=r"(?:clingo-lg=c-r=nomin-l=f-smpc=t:no_le|clingo-lg=c-r=ijcai-l=f-smpc=t:no_le)")


    \begin{tabular}{llrrrrrrr}
    \toprule
    &  & \textbf{WGC} & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{TSP} & \textbf{CA} & \textbf{Total}\\
    \midrule
    & \#Instances & 300 & 350 & 100 & 5 & 300 & 340 & 1395\\
    \midrule
    \multirow{2}{*}{\rotatebox{90}{\textsc{amo}}}
     & amoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le & 120 & 307 & \textbf{77} & \textbf{2} & \textbf{256} & \textbf{325} & 1087\\
 & amoclingo-lg=c-r=nomin-l=f-smpc=t:no\_le & \textbf{122} & \textbf{309} & 76 & \textbf{2} & 255 & 324 & \textbf{1088}\\
    \midrule
    \multirow{2}{*}{\rotatebox{90}{\textsc{eo}}} 
     & eoclingo-lg=c-r=ijcai-l=f-smpc=t:no\_le & 153 & \textbf{350} & \textbf{77} & \textbf{1} & \textbf{258} & - & 839\\
 & eoclingo-lg=c-r=nomin-l=f-smpc=t:no\_le & \textbf{157} & \textbf{350} & \textbf{77} & \textbf{1} & 256 & - & \textbf{841}\\
    \bottomrule
    \end{tabular}
    


In [103]:
create_paper_table(executables=r"(?:clingo-lg=c-r=nomin-l=f-smpc=t:no_le|clingo-lg=c-r=minfly-l=f-smpc=t:(?:no_le|minflyopt))")


    \begin{tabular}{llrrrrrrr}
    \toprule
    &  & \textbf{WGC} & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{TSP} & \textbf{CA} & \textbf{Total}\\
    \midrule
    & \#Instances & 300 & 350 & 100 & 5 & 300 & 340 & 1395\\
    \midrule
    \multirow{2}{*}{\rotatebox{90}{\textsc{amo}}}
     & amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-no\_le & \textbf{127} & 305 & 75 & 1 & \textbf{255} & \textbf{339} & \textbf{1102}\\
 & amoclingo-lg=c-r=nomin-l=f-smpc=t:no\_le & 122 & \textbf{309} & \textbf{76} & \textbf{2} & \textbf{255} & 324 & 1088\\
    \midrule
    \multirow{2}{*}{\rotatebox{90}{\textsc{eo}}} 
     & eoclingo-lg=c-r=minfly-l=f-smpc=t:no\_le & 131 & \textbf{350} & 74 & 0 & 255 & - & 810\\
 & eoclingo-lg=c-r=nomin-l=f-smpc=t:no\_le & \textbf{157} & \textbf{350} & \textbf{77} & \textbf{1} & \textbf{256} & - & \textbf{841}\\
    \bottomrule
    \end{tabular}
    


In [104]:
create_paper_table(executables=r"(?:clingo-lg=c-r=nomin-l=\w+-smpc=t:no_le|clingo-lg=c-r=minfly-l=\w+-smpc=t:(?:no_le|minflyopt))")



    \begin{tabular}{llrrrrrrr}
    \toprule
    &  & \textbf{WGC} & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{TSP} & \textbf{CA} & \textbf{Total}\\
    \midrule
    & \#Instances & 300 & 350 & 100 & 5 & 300 & 340 & 1395\\
    \midrule
    \multirow{6}{*}{\rotatebox{90}{\textsc{amo}}}
     & amoclingo-lg=c-r=minfly-l=f-smpc=t:minflyopt-no\_le & 127 & 305 & 75 & 1 & 255 & \textbf{339} & 1102\\
 & amoclingo-lg=c-r=minfly-l=h-smpc=t:minflyopt-no\_le & 126 & 309 & 75 & 1 & 255 & 338 & 1104\\
 & amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt-no\_le & \textbf{183} & \textbf{314} & 75 & 1 & \textbf{275} & \textbf{339} & \textbf{1187}\\
 & amoclingo-lg=c-r=nomin-l=f-smpc=t:no\_le & 122 & 309 & 76 & 2 & 255 & 324 & 1088\\
 & amoclingo-lg=c-r=nomin-l=h-smpc=t:no\_le & 122 & 308 & 76 & 2 & 255 & 323 & 1086\\
 & amoclingo-lg=c-r=nomin-l=t-smpc=t:no\_le & 165 & 313 & \textbf{77} & \textbf{3} & 274 & 330 & 1162\\
    \midrule
    \multirow{6}{*}{\rotatebox{90}{\textsc{eo}}} 
     & eoclingo-lg

In [105]:
create_paper_table(executables=r"(?:^clingo|^wasp|(?:eo|amo)wasp.*base|clingo-lg=c-r=(?:nomin|minfly)-l=t-smpc=t:(?:no_le|minflyopt))")


    \begin{tabular}{llrrrrrrr}
    \toprule
    &  & \textbf{WGC} & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{TSP} & \textbf{CA} & \textbf{Total}\\
    \midrule
    & \#Instances & 300 & 350 & 100 & 5 & 300 & 340 & 1395\\
    \midrule
    \multirow{5}{*}{\rotatebox{90}{\textsc{amo}}}
     & amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt-no\_le & \textbf{183} & \textbf{314} & 75 & 1 & \textbf{275} & \textbf{339} & \textbf{1187}\\
 & amoclingo-lg=c-r=nomin-l=t-smpc=t:no\_le & 165 & 313 & \textbf{77} & 3 & 274 & 330 & 1162\\
 & amowasp-base-lg=py:no\_le & 64 & 65 & 71 & 0 & 131 & 306 & 637\\
 & clingo-amo & 60 & 238 & 48 & \textbf{4} & 128 & 252 & 730\\
 & wasp-amo & 60 & 57 & 14 & 0 & 125 & 173 & 429\\
    \midrule
    \multirow{5}{*}{\rotatebox{90}{\textsc{eo}}} 
     & eoclingo-lg=c-r=minfly-l=t-smpc=t:no\_le & 169 & \textbf{350} & 75 & 1 & 262 & - & 857\\
 & eoclingo-lg=c-r=nomin-l=t-smpc=t:no\_le & \textbf{172} & \textbf{350} & \textbf{77} & 3 & \textbf{274} & - & \textbf{876}

In [106]:
create_paper_table(executables=r"(?:clingo-lg=c-r=(?:nomin|minfly)-l=t-smpc=t:(?:no_le|minflyopt-no_le)|clingo-lg=c-r=(?:nomin|minfly)-l=t-smpc=t:le_enc)", benchmarks=["GA","KC","NS"])


    \begin{tabular}{llrrrr}
    \toprule
    &  & \textbf{GA} & \textbf{KC} & \textbf{NS} & \textbf{Total}\\
    \midrule
    & \#Instances & 350 & 100 & 5 & 1395\\
    \midrule
    \multirow{4}{*}{\rotatebox{90}{\textsc{amo}}}
     & amoclingo-lg=c-r=minfly-l=t-smpc=t:minflyopt-no\_le & \textbf{314} & 75 & 1 & 390\\
 & amoclingo-lg=c-r=minfly-l=t-smpc=t:le\_enc & 312 & 75 & 1 & 388\\
 & amoclingo-lg=c-r=nomin-l=t-smpc=t:le\_enc & 313 & 75 & 2 & 390\\
 & amoclingo-lg=c-r=nomin-l=t-smpc=t:no\_le & 313 & \textbf{77} & \textbf{3} & \textbf{393}\\
    \midrule
    \multirow{4}{*}{\rotatebox{90}{\textsc{eo}}} 
     & eoclingo-lg=c-r=minfly-l=t-smpc=t:le\_enc & \textbf{350} & \textbf{86} & 1 & \textbf{437}\\
 & eoclingo-lg=c-r=nomin-l=t-smpc=t:le\_enc & \textbf{350} & 77 & 2 & 429\\
 & eoclingo-lg=c-r=minfly-l=t-smpc=t:no\_le & \textbf{350} & 75 & 1 & 426\\
 & eoclingo-lg=c-r=nomin-l=t-smpc=t:no\_le & \textbf{350} & 77 & \textbf{3} & 430\\
    \bottomrule
    \end{tabular}
    
